In [1]:
!pip install groq

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\beimn\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


  Using cached groq-1.7.0-py3-none-any.whl.metadata (16 kB)
  Using cached anyio-4.15.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.8.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.4-py3-none-any.whl.metadata (2.6 kB)
Using cached groq-1.7.0-py3-none-any.whl (143 kB)
Using cached anyio-4.15.1-py3-none-any.whl (132 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)
Using cached httpx-0.28.1-py3-none-any.whl (73 kB)
Using cached httpcore-1.0.9-py3-none-any.whl (78 kB)
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   -------------------

In [ ]:
#Set Groq API token
import os
os.environ['GROQ_API_KEY'] = groq_api_key

## Define Tools

In [ ]:
from geopy.geocoders import Nominatim
import requests

def get_weather(city: str) -> int:
    """ Provide weather information with city as input and temperature as output in degree celsius(integer) """

    #api call

    try:
        # convert city name to latitude and longitude
        geo_locator = Nominatim(user_agent="weather_agent_colab")
        location = geo_locator.geocode(city)
        if not location:
            raise ValueError("City not found")

        lat, lon = location.latitude, location.longitude

        # fetch weather data from open-meteo free API
        url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true"
        response = requests.get(url)
        response.raise_for_status()
        data = response.json()

        # extract temperature
        temperature = int(round(data["current_weather"]["temperature"]))
        return temperature
        
    except Exception as e:
        print(f"Error fetching weather: {e}")
        return 0

In [ ]:
# Create all TOOLS
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "weather",
            "description": (
                "Allows to get weather of a city in degree celsius"
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The name of the city",
                    }
                },
                "required": ["city"]
            },
        },
    }
]

In [ ]:
# Tool functions - maps strings to actual functions
TOOL_FUNCTION = {
    "weather": get_weather
}

## ReAct Loop (Tool Calling Loop)

In [ ]:
#Question
QUESTION = "How is the weather in Addis Ababa"

In [ ]:
# Create messages
MESSAGES = [
    {
        "role": "user", "content": QUESTION
    }
]

In [ ]:
# Call LLM with TOOL
from groq import Groq
client = Groq()
MODEL = "openai/gpt-oss-120b"

res = client.chat.completions.create(
    model = MODEL,
    max_tokens = 1000,
    tools = TOOLS,
    messages = MESSAGES
)

In [ ]:
MESSAGES.append(res.choices[0].message)

In [ ]:
# Execute the tool to send back observation to model

import json
for call in res.choices[0].message.tool_calls:
    tool_id =call.id
    func_name =call.function.name
    _args = json.loads(call.function.arguments)

    tool_output = TOOL_FUNCTION[func_name](**_args)
    MESSAGES.append(
        {
             "role": "tool", "content": str(tool_output), "tool_call_id": tool_id, "name": func_name
        }
    )


In [ ]:
# call llm for processing the output

from groq import Groq
client = Groq()
MODEL = "openai/gpt-oss-120b"

res = client.chat.completions.create(
    model = MODEL,
    max_tokens = 1000,
    tools = TOOLS,
    messages = MESSAGES
)

In [ ]:
clean_text = res.choices[0].message.content.replace('\u202f', ' ')
print(clean_text)